# **Algoritmos de Roteamento e Busca**
<font size=3>
    
O objetivo desta fase é pegar a coordenada cartesiana $(X, Y)$ estimada no motor IPS (Fase 2) e o mapa estruturado em grafos (Fase 1) para calcular a rota mais curta, segura e acessível até o destino desejado, traduzindo essa rota em uma sequência de passos orientados.


### **1. Estruturação do Grafo em Memória (Lista de Adjacência):**
<font size=3>

Para que os algoritmos de busca consigam percorrer o mapa com eficiência, as listas de `No` e `Aresta` carregadas do JSON na Fase 1 devem ser convertidas em uma **Lista de Adjacência**.

* **Representação em Kotlin:** Usamos um mapa (`Map<String, List<Aresta>>`) onde a chave é o `id` do nó de origem e o valor é a lista de caminhos que saem daquele nó.

* **Filtro Dinâmico de Acessibilidade:** Durante a montagem da lista de adjacência, arestas que contêm obstáculos impeditivos para pessoas com deficiência visual (ex: escadas sem corrimão ou áreas em obra) podem ter seu peso (distância) aumentado drasticamente ou ser completamente desconsideradas do cálculo.

In [ ]:
class NavigableGraph(private val nodesMap: Map<String, Node>, edges: List<Edge>) {
    // Lista de adjacência: Nó de Origem -> Lista de Arestas que saem dele
    val adjacencyList: Map<String, List<Edge>> = edges
        .filter { it.accessibleVisualImpairment } // Filtra apenas caminhos acessíveis
        .groupBy { it.originNodeId }

    fun getNode(id: String): Node? = nodesMap[id]
}

### **2. Algoritmo de Menor Caminho (Dijkstra / $A^*$):**
<font size=3>

O algoritmo de **Dijkstra** encontra a rota de menor custo calculando a menor soma de distâncias físicas entre os nós. O algoritmo **$A^*$ (A-Star)** melhora o desempenho usando uma **heurística** (geralmente a distância em linha reta até o destino final, ou distância euclidiana) para priorizar os nós na direção correta.

**Passo a Passo do Dijkstra:**
1. Atribuir a todos os nós uma distância inicial infinita, exceto ao nó de partida (distância = 0).
2. Manter uma fila de prioridade (`PriorityQueue`) dos nós não visitados ordenada pela menor distância acumulada.
3. Para o nó atual, examinar todos os seus vizinhos acessíveis e calcular a distância acumulada até eles.
4. Se a nova distância for menor que a registrada anteriormente, atualizar o valor e registrar o "pai" do nó para permitir a reconstrução do caminho no final.

In [ ]:
import java.util.PriorityQueue

In [ ]:
data class CalculatedPath(val pathNodes: List<Node>,
                          val totalDistanceMeters: Double)

fun calculateDijkstraRoute(graph: NavigableGraph, originId: String, destinationId: String): CalculatedPath? {
    val distances = mutableMapOf<String, Double>().withDefault { Double.POSITIVE_INFINITY }
    val predecessors = mutableMapOf<String, String?>()
    val priorityQueue = PriorityQueue<Pair<String, Double>>(compareBy { it.second })

    distances[originId] = 0.0
    priorityQueue.add(Pair(originId, 0.0))

    while (priorityQueue.isNotEmpty()) {
        val (currentNodeId, currentDist) = priorityQueue.poll()!!

        if (currentNodeId == destinationId) break // Destino encontrado!

        if (currentDist > distances.getValue(currentNodeId)) continue

        val neighbors = graph.adjacencyList[currentNodeId] ?: emptyList()
        for (edge in neighbors) {
            val newDist = currentDist + edge.distanceMeters
            val neighborNodeId = edge.destinationNodeId

            if (newDist < distances.getValue(neighborNodeId)) {
                distances[neighborNodeId] = newDist
                predecessors[neighborNodeId] = currentNodeId
                priorityQueue.add(Pair(neighborNodeId, newDist))
            }
        }
    }

    // Reconstrução do caminho de trás para frente
    val pathNodes = mutableListOf<Node>()
    var current: String? = destinationId
    
    while (current != null) {
        val node = graph.getNode(current) ?: return null
        pathNodes.add(0, node)
        current = predecessors[current]
    }

    if (pathNodes.firstOrNull()?.id != originId) return null // Caminho inalcançável

    return CalculatedPath(pathNodes, distances.getValue(destinationId))
}

### **3. Geometria de Orientação e Mudança de Direção:**
<font size=3>

Saber apenas a lista de pontos não é suficiente. Para instruir a pessoa ("Vire à esquerda", "Siga em frente"), precisamos calcular a mudança de ângulo ($\Delta \theta$) entre três nós consecutivos na sequência da rota: $N_1 \to N_2 \to N_3$.

1. **Vetor 1 ($\vec{v_1}$):** Deslocamento do nó $N_1(x_1, y_1)$ para $N_2(x_2, y_2)$.

2. **Vetor 2 ($\vec{v_2}$):** Deslocamento do nó $N_2(x_2, y_2)$ para $N_3(x_3, y_3)$.

3. **Cálculo dos Ângulos de Direção:**

    $$\theta_1 = \text{atan2}(y_2 - y_1, x_2 - x_1)$$
    
    $$\theta_2 = \text{atan2}(y_3 - y_2, x_3 - x_2)$$
    
    $$\Delta \theta = (\theta_2 - \theta_1) \text{ convertido para graus em } [-180^\circ, 180^\circ]$$

**Classificação dos Comandos Verbais:**
* **$-20^\circ \le \Delta \theta \le 20^\circ$:** "Siga em frente"
* **$20^\circ < \Delta \theta \le 60^\circ$:** "Ligeira curva à direita"
* **$60^\circ < \Delta \theta \le 120^\circ$:** "Vire à direita"
* **$-60^\circ \le \Delta \theta < -20^\circ$:** "Ligeira curva à esquerda"
* **$-120^\circ \le \Delta \theta < -60^\circ$:** "Vire à esquerda"
* **$\vert{}\Delta \theta\vert{} > 120^\circ$:** "Faça meia-volta"

In [ ]:
import kotlin.math.atan2

In [ ]:
fun calculateDirectionInstruction(n1: Node, n2: Node, n3: Node): String {
    val angle1 = Math.toDegrees(atan2(n2.y - n1.y, n2.x - n1.x))
    val angle2 = Math.toDegrees(atan2(n3.y - n2.y, n3.x - n2.x))

    var delta = angle2 - angle1
    // Normalização do ângulo para o intervalo [-180, 180]
    while (delta > 180) delta -= 360
    while (delta < -180) delta += 360

    return when {
        delta in -20.0..20.0 -> "Siga em frente"
        delta in 20.0..60.0 -> "Ligeira curva à direita"
        delta in 60.0..120.0 -> "Vire à direita"
        delta in -60.0..-20.0 -> "Ligeira curva à esquerda"
        delta in -120.0..-60.0 -> "Vire à esquerda"
        else -> "Faça meia-volta"
    }
}

### **4. Geração da Rota com Instruções Narráveis:**
<font size=3>

Consiste em processar o `CaminhoCalculado` e agrupar segmentos retilíneos, informando a distância aproximada que o usuário deve percorrer antes de realizar a próxima conversão.

In [ ]:
data class NavigationInstruction(val voiceCommand: String, 
                                 val approximateDistanceMeters: Double,
                                 val targetNode: Node)

fun generateStepByStepGuide(path: CalculatedPath): List<NavigationInstruction> {
    val nodes = path.pathNodes
    if (nodes.size < 2) return emptyList()

    val instructions = mutableListOf<NavigationInstruction>()
    
    for (i in 0 until nodes.size - 2) {
        val n1 = nodes[i]
        val n2 = nodes[i + 1]
        val n3 = nodes[i + 2]

        val command = calculateDirectionInstruction(n1, n2, n3)
        val sectionDist = Math.hypot(n2.x - n1.x, n2.y - n1.y)

        instructions.add(NavigationInstruction(command, sectionDist, n2))
    }

    return instructions
}

### **Atividade prática:**
<font size=3>
    
Traçar a rota e imprimir a narração de um corredor:\
Escreva um script para montar um grafo fictício com 4 pontos no formato de "L" no corredor da faculdade e calcular a navegação do Ponto A ao Ponto D.


<details>
<summary>Resposta:</summary>

<font size=2>
<br>
    
```kotlin
// 1. Criar os Nós do Grafo (Mapa simplificado em formato de L)
val nA = Node("A", "Entrada Bloco", 0.0, 0.0, 0)
val nB = Node("B", "Corredor Central", 0.0, 10.0, 0)
val nC = Node("C", "Curva da Biblioteca", 0.0, 20.0, 0) // Segue reto
val nD = Node("D", "Porta do Laboratório", 10.0, 20.0, 0) // Curva de 90° à direita

val nodesMap = listOf(nA, nB, nC, nD).associateBy { it.id }

// 2. Criar as Arestas
val edges = listOf(Edge("A", "B", 10.0),
                   Edge("B", "C", 10.0),
                   Edge("C", "D", 10.0))

val graph = NavigableGraph(nodesMap, edges)

// 3. Executar o Algoritmo de Busca (Dijkstra)
val route = calculateDijkstraRoute(graph, originId = "A", destinationId = "D")

if (route != null) {
    println("=== ROTA ENCONTRADA ===")
    println("Distância Total: %.1f metros".format(route.totalDistanceMeters))
    println("Sequência de Nós: ${route.pathNodes.map { it.name }}\n")

    // 4. Gerar e Imprimir as Instruções de Direção
    val instructions = generateStepByStepGuide(route)
    println("=== INSTRUÇÕES PARA A SÍNTESE DE VOZ ===")
    for ((index, inst) in instructions.withIndex()) {
        println("Passo ${index + 1}: Em %.0f metros na direção de '${inst.targetNode.name}', %s."
            .format(inst.approximateDistanceMeters, inst.voiceCommand.lowercase()))
    }
} 
else {
    println("Não foi possível encontrar uma rota acessível entre os pontos selecionados.")
}

```
<br>

</details>
